TOOLS


Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)

2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. I know they\'re birds, and some species are known for mimicking human speech. But why do they do that? Maybe it\'s related to their environment or their social behavior.\n\nFirst, I should think about their natural habitat. Parrots are social animals, right? They live in flocks. In the wild, they might use sounds to communicate with each other, like to signal danger or to find each other. If they\'re social, maybe imitating sounds is a way to bond or to fit in with the group. So in a domestic setting, maybe they mimic human speech because humans are their flock now.\n\nAnother thought: vocal imitation as a survival mechanism. In the wild, if a parrot can mimic other sounds, maybe it helps them avoid predators or communicate in complex environments. But how does that translate to talking? Maybe they learn to imitate human speech through interaction, especially

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'fg9n16fqq', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.150379199, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.007253152, 'prompt_tokens_details': None, 'queue_time': 0.050579937, 'total_time': 0.157632351}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f04b0-b36b-7172-9

Tool Execution Loop

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Boston is the location here, so I need to call that function with "Boston" as the argument. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': '7b2svcckg', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 153, 'total_tokens': 250, 'completion_time': 0.146972377, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.007098465, 'prompt_tokens_details': None, 'queue_time': 0.056690851, 'total_time': 0.154070842}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 